In [ ]:
# Cell 1: Check GPU (set Runtime -> Change runtime type -> T4 GPU)
import numpy as np
import torch
import torch.nn as nn

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", dev)

Using device: cpu


# Cell 2: Simulate simple soil data (sand %, silt %, clay %, pH, organic matter)
rng = np.random.default_rng(0)

# 200 rows across 4 soil types: 0=Clay, 1=Sand, 2=Loam, 3=Silt
X_raw = rng.uniform(low=[10, 10, 5, 5.5, 1.0], high=[80, 70, 60, 7.5, 6.0], size=(200, 5)).astype(np.float32)
y_raw = rng.integers(0, 4, size=(200,))

X = torch.tensor(X_raw, device=dev)
y = torch.tensor(y_raw, dtype=torch.long, device=dev)

In [ ]:
# Cell 3: MLP with two hidden layers (no softmax before CrossEntropyLoss)
model = nn.Sequential(
    nn.Linear(5, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 4),
).to(dev)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Cell 4: Train and print accuracy
for epoch in range(1, 101):
    optimizer.zero_grad()
    scores = model(X)
    loss = loss_fn(scores, y)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        acc = (scores.argmax(dim=1) == y).float().mean().item()
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.2%}")

Epoch  20 | Loss: 1.3535 | Accuracy: 28.00%
Epoch  40 | Loss: 1.3394 | Accuracy: 27.00%
Epoch  60 | Loss: 1.3188 | Accuracy: 32.50%
Epoch  80 | Loss: 1.3043 | Accuracy: 33.50%
Epoch 100 | Loss: 1.2974 | Accuracy: 34.00%
